# 01 — Data Audit

## Objective

Inspect the raw manufacturing datasets before any transformation or analysis.

The audit covers:

- dataset dimensions
- column structure
- data types
- missing values
- duplicate records
- date ranges
- categorical values
- basic validity checks

No cleaning or feature engineering is performed in this notebook.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Detect project root whether notebook is run
# from the project root or from /notebooks

cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DIR

WindowsPath('c:/Users/erenk/OneDrive/Desktop/manufacturing-oee-downtime-reliability/data/raw')

In [3]:
files = {
    "oee_before": RAW_DIR / "OEEdataset.csv",
    "downtime_before": RAW_DIR / "DowntimeDataset.csv",
    "oee_after": RAW_DIR / "OEEdataset_afterLSS.csv",
    "downtime_after": RAW_DIR / "DowntimeDataset_afterLSS.csv",
}

for name, path in files.items():
    print(f"{name:20} -> {path.exists()} | {path.name}")

oee_before           -> True | OEEdataset.csv
downtime_before      -> True | DowntimeDataset.csv
oee_after            -> True | OEEdataset_afterLSS.csv
downtime_after       -> True | DowntimeDataset_afterLSS.csv


In [4]:
oee_before = pd.read_csv(files["oee_before"])
downtime_before = pd.read_csv(files["downtime_before"])

oee_after = pd.read_csv(files["oee_after"])
downtime_after = pd.read_csv(files["downtime_after"])

C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\2094318966.py:5: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  downtime_after = pd.read_csv(files["downtime_after"])


In [5]:
datasets = {
    "OEE Before LSS": oee_before,
    "Downtime Before LSS": downtime_before,
    "OEE After LSS": oee_after,
    "Downtime After LSS": downtime_after,
}

In [6]:
overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
    })

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns,duplicate_rows,missing_cells
0,OEE Before LSS,65,13,2,39
1,Downtime Before LSS,635,10,0,25
2,OEE After LSS,43,13,0,0
3,Downtime After LSS,3022,1012,2481,3052864


In [7]:
# Remove completely empty columns created by trailing delimiters
for name, df in datasets.items():
    datasets[name] = df.dropna(axis=1, how="all")

oee_before = datasets["OEE Before LSS"]
downtime_before = datasets["Downtime Before LSS"]
oee_after = datasets["OEE After LSS"]
downtime_after = datasets["Downtime After LSS"]

In [10]:
overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
    })

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns,duplicate_rows,missing_cells
0,OEE Before LSS,62,13,0,0
1,Downtime Before LSS,635,10,0,25
2,OEE After LSS,43,13,0,0
3,Downtime After LSS,540,10,0,0


In [9]:
# Remove completely empty columns and rows caused by CSV export formatting

for name, df in datasets.items():
    datasets[name] = (
        df
        .dropna(axis=1, how="all")
        .dropna(axis=0, how="all")
        .reset_index(drop=True)
    )

oee_before = datasets["OEE Before LSS"]
downtime_before = datasets["Downtime Before LSS"]
oee_after = datasets["OEE After LSS"]
downtime_after = datasets["Downtime After LSS"]

In [11]:
for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nData types:")
    print(df.dtypes)


OEE Before LSS

Columns:
['shiftDate', 'OEE', 'Availability', 'Performance', 'Quality', 'Sumof goodProductionTimeSec', 'Sum of slowProductionTimeSec', 'Sumof plannedTime', 'Sumof idealProductionQuantity', 'Sum of plannedStopsTimeSec', 'Sum of unplannedStopsTimeSec', 'Sum of goodQty', 'Sum of totalQty']

Data types:
shiftDate                         object
OEE                              float64
Availability                     float64
Performance                      float64
Quality                          float64
Sumof goodProductionTimeSec      float64
Sum of slowProductionTimeSec     float64
Sumof plannedTime                float64
Sumof idealProductionQuantity    float64
Sum of plannedStopsTimeSec       float64
Sum of unplannedStopsTimeSec     float64
Sum of goodQty                   float64
Sum of totalQty                  float64
dtype: object

Downtime Before LSS

Columns:
['Date', 'Productcode', 'StopGroup', 'Stop', 'StopType', 'StopLocation', 'ExtraText', 'StopStartTime', '

In [12]:
missing_summary = []

for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    
    for column, count in missing.items():
        missing_summary.append({
            "dataset": name,
            "column": column,
            "missing_count": count,
            "missing_pct": round(count / len(df) * 100, 2)
        })

missing_df = pd.DataFrame(missing_summary)

missing_df

,dataset,column,missing_count,missing_pct
0,Downtime Before LSS,ExtraText,25,3.94


In [13]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head(3))


OEE Before LSS


,shiftDate,OEE,Availability,Performance,Quality,Sumof goodProductionTimeSec,Sum of slowProductionTimeSec,Sumof plannedTime,Sumof idealProductionQuantity,Sum of plannedStopsTimeSec,Sum of unplannedStopsTimeSec,Sum of goodQty,Sum of totalQty
0,9/22/25,0.8254,0.8902,0.9272,0.9996,14604.0,1317.0,17884.0,38626.75,13111.0,668.0,35800.0,35815.0
1,9/23/25,0.5891,0.6236,0.9447,0.9975,17398.0,1123.0,29700.0,27961.00,634.0,10545.0,26350.0,26416.0
2,9/24/25,0.7498,0.7833,0.9572,0.9951,21036.0,1524.0,28800.0,22560.00,3234.0,3006.0,21490.0,21594.0



Downtime Before LSS


,Date,Productcode,StopGroup,Stop,StopType,StopLocation,ExtraText,StopStartTime,StopEndTime,StopDuration(min)
0,9/22/25,KK200-1,OPERATIONAL,NO PRODUCTION SCHEDULE,Planned - Not included in OEE,EXTRUDER PRESS,-,2025-09-22 07:15,2025-09-22 10:31,196.9
1,9/22/25,KK200-1,SETUPS-CHANGEOVERS,WIRE ROLL REPLACEMENT,Unplanned,CUTTER,NaN,2025-09-22 10:58,2025-09-22 11:01,3.8
2,9/22/25,KK200-1,MATERIALS,MANUAL BRICK HANDLING,Unplanned,HACKING STATION,-,2025-09-22 11:19,2025-09-22 11:26,7.4



OEE After LSS


,shiftDate,OEE,Availability,Performance,Quality,Sum of goodProdction,Sum of slowProduction,Sum of plannedTime,Sum of idealQty,Sum of plannedStopsTimeSec,Sum of unplannedStopsTimeSec,Sum of goodQty,Sum of totalQty
0,1/30/26,0.644347,0.665627,0.976219,0.991612,17587,984,27900,29309.0000,3178,6048,28372,28612
1,2/2/26,0.687063,0.834040,0.916115,0.899207,22693,2078,29700,65357.5000,2276,2653,53840,59875
2,2/3/26,0.609544,0.615118,1.022932,0.968724,17517,752,29700,35351.3334,3487,7944,35031,36162



Downtime After LSS


,Date,Product code,Stop group,Stop,Stop type,Stop Location,Extra text,Stop start time,Stop end time,Stop duration (min) (TTR)
0,1/30/26,KK250,OPERATIONAL,CLEANING,Planned - Included in OEE,EXTRUDER PRESS,-,2026-01-30 07:15,2026-01-30 07:23,9.0
1,1/30/26,KK250,MATERIALS,MANUAL BRICK HANDLING,Unplanned,HACKING STATION,-,2026-01-30 07:54,2026-01-30 07:58,4.0
2,1/30/26,9KK25,SETUPS-CHANGEOVERS,MATERIAL CHANGEOVER,Planned - Included in OEE,EXTRUDER PRESS,-,2026-01-30 08:44,2026-01-30 09:07,22.0


In [14]:
observation_check = pd.DataFrame([
    {
        "dataset": "OEE Before LSS",
        "rows": len(oee_before),
        "unique_dates": oee_before["shiftDate"].nunique(),
        "duplicate_dates": oee_before["shiftDate"].duplicated().sum()
    },
    {
        "dataset": "OEE After LSS",
        "rows": len(oee_after),
        "unique_dates": oee_after["shiftDate"].nunique(),
        "duplicate_dates": oee_after["shiftDate"].duplicated().sum()
    },
    {
        "dataset": "Downtime Before LSS",
        "rows": len(downtime_before),
        "unique_dates": downtime_before["Date"].nunique(),
        "duplicate_dates": downtime_before["Date"].duplicated().sum()
    },
    {
        "dataset": "Downtime After LSS",
        "rows": len(downtime_after),
        "unique_dates": downtime_after["Date"].nunique(),
        "duplicate_dates": downtime_after["Date"].duplicated().sum()
    }
])

observation_check

,dataset,rows,unique_dates,duplicate_dates
0,OEE Before LSS,62,62,0
1,OEE After LSS,43,43,0
2,Downtime Before LSS,635,62,573
3,Downtime After LSS,540,45,495


In [15]:
before_event_duplicates = downtime_before.duplicated(
    subset=[
        "StopStartTime",
        "StopEndTime",
        "Stop",
        "StopLocation"
    ]
).sum()

after_event_duplicates = downtime_after.duplicated(
    subset=[
        "Stop start time",
        "Stop end time",
        "Stop",
        "Stop Location"
    ]
).sum()

pd.DataFrame({
    "dataset": ["Downtime Before LSS", "Downtime After LSS"],
    "duplicate_event_keys": [
        before_event_duplicates,
        after_event_duplicates
    ]
})

,dataset,duplicate_event_keys
0,Downtime Before LSS,0
1,Downtime After LSS,0


In [16]:
# Convert date columns temporarily for audit purposes

oee_before_dates = pd.to_datetime(oee_before["shiftDate"], errors="coerce")
oee_after_dates = pd.to_datetime(oee_after["shiftDate"], errors="coerce")

downtime_before_dates = pd.to_datetime(downtime_before["Date"], errors="coerce")
downtime_after_dates = pd.to_datetime(downtime_after["Date"], errors="coerce")


date_coverage = pd.DataFrame([
    {
        "dataset": "OEE Before LSS",
        "start_date": oee_before_dates.min(),
        "end_date": oee_before_dates.max(),
        "unique_dates": oee_before_dates.nunique(),
        "invalid_dates": oee_before_dates.isna().sum()
    },
    {
        "dataset": "OEE After LSS",
        "start_date": oee_after_dates.min(),
        "end_date": oee_after_dates.max(),
        "unique_dates": oee_after_dates.nunique(),
        "invalid_dates": oee_after_dates.isna().sum()
    },
    {
        "dataset": "Downtime Before LSS",
        "start_date": downtime_before_dates.min(),
        "end_date": downtime_before_dates.max(),
        "unique_dates": downtime_before_dates.nunique(),
        "invalid_dates": downtime_before_dates.isna().sum()
    },
    {
        "dataset": "Downtime After LSS",
        "start_date": downtime_after_dates.min(),
        "end_date": downtime_after_dates.max(),
        "unique_dates": downtime_after_dates.nunique(),
        "invalid_dates": downtime_after_dates.isna().sum()
    }
])

date_coverage

C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1633737280.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oee_before_dates = pd.to_datetime(oee_before["shiftDate"], errors="coerce")
C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1633737280.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oee_after_dates = pd.to_datetime(oee_after["shiftDate"], errors="coerce")
C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1633737280.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  downtime_before_dates = pd.to_datetime(downtime_before["Date"], errors="coerce")
C:\Users\e

,dataset,start_date,end_date,unique_dates,invalid_dates
0,OEE Before LSS,2025-09-22,2025-12-18,62,0
1,OEE After LSS,2026-01-30,2026-04-03,43,0
2,Downtime Before LSS,2025-09-22,2025-12-18,62,0
3,Downtime After LSS,2026-01-30,2026-04-03,45,0


In [17]:
# Standardize dates temporarily for alignment checks

oee_before_date_set = set(
    pd.to_datetime(oee_before["shiftDate"]).dt.normalize()
)

downtime_before_date_set = set(
    pd.to_datetime(downtime_before["Date"]).dt.normalize()
)

oee_after_date_set = set(
    pd.to_datetime(oee_after["shiftDate"]).dt.normalize()
)

downtime_after_date_set = set(
    pd.to_datetime(downtime_after["Date"]).dt.normalize()
)


date_alignment = {
    "before_downtime_not_in_oee":
        sorted(downtime_before_date_set - oee_before_date_set),

    "before_oee_not_in_downtime":
        sorted(oee_before_date_set - downtime_before_date_set),

    "after_downtime_not_in_oee":
        sorted(downtime_after_date_set - oee_after_date_set),

    "after_oee_not_in_downtime":
        sorted(oee_after_date_set - downtime_after_date_set),
}

for key, dates in date_alignment.items():
    print(f"\n{key}")
    print(dates)


before_downtime_not_in_oee
[]

before_oee_not_in_downtime
[]

after_downtime_not_in_oee
[Timestamp('2026-03-25 00:00:00'), Timestamp('2026-04-01 00:00:00')]

after_oee_not_in_downtime
[]


C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1227527868.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(oee_before["shiftDate"]).dt.normalize()
C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1227527868.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(downtime_before["Date"]).dt.normalize()
C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1227527868.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(oee_after["shiftDate"]).dt.normalize()
C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\1227527868.py:16: UserWarn

In [19]:
extra_after_dates = pd.to_datetime(
    ["2026-03-25", "2026-04-01"]
)

downtime_after_temp = downtime_after.copy()
downtime_after_temp["Date_parsed"] = pd.to_datetime(
    downtime_after_temp["Date"],
    errors="coerce"
).dt.normalize()

extra_date_events = downtime_after_temp[
    downtime_after_temp["Date_parsed"].isin(extra_after_dates)
]

extra_date_events

C:\Users\erenk\AppData\Local\Temp\ipykernel_23528\2573549439.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  downtime_after_temp["Date_parsed"] = pd.to_datetime(


,Date,Product code,Stop group,Stop,Stop type,Stop Location,Extra text,Stop start time,Stop end time,Stop duration (min) (TTR),Date_parsed
460,3/25/26,7KK25,OPERATIONAL,NO PRODUCTION SCHEDULE,Planned - Not included in OEE,-,-,2026-03-25 07:15,2026-03-25 15:15,480.0,2026-03-25
508,4/1/26,7KK25,OPERATIONAL,NO PRODUCTION SCHEDULE,Planned - Not included in OEE,-,-,2026-04-01 15:30,2026-04-01 21:00,330.0,2026-04-01


### Date Alignment Finding

The post-LSS downtime dataset contains two dates that are not present in the post-LSS OEE dataset:

- 2026-03-25
- 2026-04-01

These dates were retained for investigation rather than removed, as the mismatch may reflect non-production days, planned shutdowns, or missing OEE records.

In [20]:
downtime_after_temp["Date_parsed"] = pd.to_datetime(
    downtime_after_temp["Date"],
    format="%m/%d/%y",
    errors="coerce"
)

In [21]:
# Downtime duration consistency check

dt_before = downtime_before.copy()
dt_after = downtime_after.copy()

dt_before["start_dt"] = pd.to_datetime(
    dt_before["StopStartTime"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

dt_before["end_dt"] = pd.to_datetime(
    dt_before["StopEndTime"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

dt_after["start_dt"] = pd.to_datetime(
    dt_after["Stop start time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

dt_after["end_dt"] = pd.to_datetime(
    dt_after["Stop end time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)


dt_before["calculated_duration_min"] = (
    dt_before["end_dt"] - dt_before["start_dt"]
).dt.total_seconds() / 60

dt_after["calculated_duration_min"] = (
    dt_after["end_dt"] - dt_after["start_dt"]
).dt.total_seconds() / 60

In [22]:
duration_check = pd.DataFrame([
    {
        "dataset": "Downtime Before LSS",
        "invalid_start": dt_before["start_dt"].isna().sum(),
        "invalid_end": dt_before["end_dt"].isna().sum(),
        "negative_duration": (dt_before["calculated_duration_min"] < 0).sum(),
        "duration_mismatch": (
            abs(
                dt_before["calculated_duration_min"]
                - dt_before["StopDuration(min)"]
            ) > 1
        ).sum()
    },
    {
        "dataset": "Downtime After LSS",
        "invalid_start": dt_after["start_dt"].isna().sum(),
        "invalid_end": dt_after["end_dt"].isna().sum(),
        "negative_duration": (dt_after["calculated_duration_min"] < 0).sum(),
        "duration_mismatch": (
            abs(
                dt_after["calculated_duration_min"]
                - dt_after["Stop duration (min) (TTR)"]
            ) > 1
        ).sum()
    }
])

duration_check

,dataset,invalid_start,invalid_end,negative_duration,duration_mismatch
0,Downtime Before LSS,0,0,0,0
1,Downtime After LSS,0,0,0,0


In [23]:
oee_checks = []

for name, df in {
    "OEE Before LSS": oee_before,
    "OEE After LSS": oee_after
}.items():
    
    temp = df.copy()
    
    temp["calculated_oee"] = (
        temp["Availability"]
        * temp["Performance"]
        * temp["Quality"]
    )
    
    temp["oee_difference"] = (
        temp["OEE"] - temp["calculated_oee"]
    ).abs()
    
    oee_checks.append({
        "dataset": name,
        "rows": len(temp),
        "max_difference": temp["oee_difference"].max(),
        "mean_difference": temp["oee_difference"].mean(),
        "mismatches_gt_0.001": (
            temp["oee_difference"] > 0.001
        ).sum()
    })

oee_formula_check = pd.DataFrame(oee_checks)

oee_formula_check

,dataset,rows,max_difference,mean_difference,mismatches_gt_0.001
0,OEE Before LSS,62,1.181476e-02,6.730046e-04,10
1,OEE After LSS,43,1.375131e-09,4.288928e-10,0


In [24]:
range_check = []

for name, df in {
    "OEE Before LSS": oee_before,
    "OEE After LSS": oee_after
}.items():
    
    range_check.append({
        "dataset": name,
        "oee_below_0": (df["OEE"] < 0).sum(),
        "oee_above_1": (df["OEE"] > 1).sum(),
        "availability_below_0": (df["Availability"] < 0).sum(),
        "availability_above_1": (df["Availability"] > 1).sum(),
        "performance_below_0": (df["Performance"] < 0).sum(),
        "performance_above_1": (df["Performance"] > 1).sum(),
        "quality_below_0": (df["Quality"] < 0).sum(),
        "quality_above_1": (df["Quality"] > 1).sum(),
    })

range_check_df = pd.DataFrame(range_check)

range_check_df

,dataset,oee_below_0,oee_above_1,availability_below_0,availability_above_1,performance_below_0,performance_above_1,quality_below_0,quality_above_1
0,OEE Before LSS,0,0,0,0,0,2,0,0
1,OEE After LSS,0,0,0,0,0,9,0,0
